# Milestone M3: Cloud GPU Training & Evaluation (Google Colab)
### Isolated Drywall Detection Module (`drywall_detection`)

This notebook trains and benchmarks the champion **SegFormer-B0** and baseline **ResNet18 U-Net** semantic segmentation models on architectural drawings using NVIDIA Tesla T4 GPU acceleration.

## 1. Environment & Hardware Probe

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Dependency Installation

In [ ]:
!pip install -q transformers==4.41.2 timm==1.0.22 pymupdf shapely opencv-python-headless psutil pillow

## 3. Clone / Mount Workspace & Verify Splits

In [ ]:
import json
from pathlib import Path

splits_summary = Path("drywall_detection/data/processed/splits/splits_summary.json")
if splits_summary.exists():
    with open(splits_summary) as f:
        summary = json.load(f)
    print("Splits Summary:", json.dumps(summary, indent=2))
else:
    print("Splits summary not found. Ensure workspace root is set.")

## 4. Run High-Throughput GPU Training

In [ ]:
from drywall_detection.models.train_colab import run_colab_training

# Execute 30 epochs with batch size 16 on GPU
run_colab_training(
    train_manifest="drywall_detection/data/processed/splits/train.json",
    val_manifest="drywall_detection/data/processed/splits/val.json",
    checkpoint_dir="drywall_detection/checkpoints/best_model",
    batch_size=16,
    num_workers=2,
    max_epochs=30,
    learning_rate=1e-4,
    early_stopping_patience=7,
)

## 5. Evaluate on Held-Out Test Set & Generate Overlays

In [ ]:
from drywall_detection.models.evaluate import evaluate_checkpoint

eval_results = evaluate_checkpoint(
    checkpoint_dir="drywall_detection/checkpoints/best_model",
    manifest_path="drywall_detection/data/processed/splits/test.json",
    save_overlays=True,
    max_overlays=5,
)
print(f"Evaluated on {eval_results['test_samples']} test samples.")
print(f"Overall mIoU: {eval_results['mean_iou_all']:.4f}")
print(f"Wall mIoU (Present): {eval_results['wall_mean_iou_present']:.4f}")

## 6. Package and Download Deliverable Bundle

In [ ]:
import shutil
shutil.make_archive("/content/drywall_best_model_bundle", "zip", "drywall_detection/checkpoints/best_model")
try:
    from google.colab import files
    files.download("/content/drywall_best_model_bundle.zip")
except Exception:
    print("Saved bundle archive to /content/drywall_best_model_bundle.zip")